In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# =============================================================================
# preferencias de usuarios
# =============================================================================

preferencias = pd.read_csv("usuarios_preferencias.csv", sep=";", header=None)
preferencias.columns = ['id_user', 'id_preferencia', 'score']

# Crear una matriz de preferencias
matriz_preferencias = preferencias.pivot(index='id_user', columns='id_preferencia', values='score')
columnas_deseadas = list(range(1, 116))
matriz_preferencias = matriz_preferencias.reindex(columns=columnas_deseadas, fill_value=0).fillna(0)
matriz_preferencias = matriz_preferencias.T

padres = pd.read_csv("preferencias.csv")
padres.index = range(1, 116)

matriz_preferencias_mayor0 = (matriz_preferencias > 0).copy()
matriz_preferencias_mayor0['padre'] = padres['parent']
# Agrupar por 'padre' (opcional, para conteos)
matriz_preferencias_conteos = matriz_preferencias_mayor0.groupby('padre').sum()

# =============================================================================
# top porcentaje de preferencias por categoría
# =============================================================================
def filtrar_top_porcentaje(df, porcentaje=0.2):
    df_filtrado = df.copy()
    columnas_usuario = [col for col in df.columns if col != 'padre']
    
    # Para cada grupo (categoría) se conservan los top valores según el porcentaje
    for padre, grupo in df.groupby('padre'):
        for col in columnas_usuario:
            conteo_puntuaciones = (grupo[col] > 0).sum()
            n_items = max(1, int(conteo_puntuaciones * porcentaje))
            if n_items > 0:
                indices_top = grupo[col].nlargest(n_items).index
                indices_no_top = grupo.index.difference(indices_top)
                df_filtrado.loc[indices_no_top, col] = 0
    return df_filtrado


matriz_preferencias.loc[:, 'padre'] = padres['parent']
matriz_filtrada = filtrar_top_porcentaje(matriz_preferencias, porcentaje=0.2)

# =============================================================================
# ítems
# =============================================================================

items_clasificacion = pd.read_csv("clasificacion_items.csv", sep=";", header=None)
items_clasificacion.columns = ['id_item', 'id_preferencia', 'score']

# Agrupar para calcular la media (en caso de duplicados) y pivotear
items_group = items_clasificacion.groupby(['id_item', 'id_preferencia'], as_index=False).mean()
matriz_items = items_group.pivot(index='id_item', columns='id_preferencia', values='score')
matriz_items = matriz_items.reindex(columns=columnas_deseadas, fill_value=0).fillna(0)

# =============================================================================
# Calcular la similitud entre ítems y las preferencias filtradas
# =============================================================================
matriz_filtradaT = matriz_filtrada.T
similitud_items = cosine_similarity(matriz_items.values, matriz_filtradaT.values)
matriz_similitud_items_df = pd.DataFrame(similitud_items, 
                                         index=matriz_items.index, 
                                         columns=matriz_filtradaT.index).T

# =============================================================================
# histórico de puntuaciones de usuarios
# =============================================================================
# Cargar nombres y visitas de ítems
items_names = pd.read_csv("items.csv", sep=";", header=None, encoding="latin-1")
items_names.columns = ['id_item', 'name_item', 'visitas']

usuarios_historico = pd.read_csv("puntuaciones_usuario_base.csv", sep=";", header=None)
usuarios_historico.columns = ['id_user', 'id_item', 'valoracion']

interacciones = pd.merge(usuarios_historico, items_names[['id_item', 'visitas']],
                         on='id_item', how='left')
interacciones['visitas'] = interacciones['visitas'].fillna(0)

interacciones['weighted_rating'] = interacciones.apply(
    lambda row: row['valoracion'] if row['valoracion'] >= 4 else -row['valoracion'],
    axis=1
)

# =============================================================================
# Crear la matriz de interacciones de usuarios e ítems
# =============================================================================
matriz_interacciones = interacciones.pivot(index='id_user', columns='id_item', values='weighted_rating').fillna(0)

# =============================================================================
# Calcular el vector de usuario agrupado por categoría (id_padre)
# =============================================================================
# Unir interacciones con clasificación para obtener id_padre y score
items_clasificacion = pd.read_csv("clasificacion_items.csv", sep=";", header=None)
items_clasificacion.columns = ['id_item', 'id_padre', 'score']
interacciones_con_padre = pd.merge(interacciones, items_clasificacion[['id_item', 'id_padre', 'score']],
                                   on='id_item', how='left')
# Distribuir el weighted_rating proporcionalmente según el score
interacciones_con_padre['total_score'] = interacciones_con_padre.groupby('id_item')['score'].transform('sum')
interacciones_con_padre['weighted_rating_final'] = interacciones_con_padre['weighted_rating'] * \
    (interacciones_con_padre['score'] / interacciones_con_padre['total_score'])

# Agrupar por usuario y categoría y pivotear
agrupado = interacciones_con_padre.groupby(['id_user', 'id_padre'])['weighted_rating_final'].sum().reset_index()
vector_por_usuario = agrupado.pivot(index='id_user', columns='id_padre', values='weighted_rating_final').fillna(0)
vector_por_usuario = vector_por_usuario.reindex(columns=columnas_deseadas, fill_value=0)

# =============================================================================
# Calcular la similitud entre el vector de usuario y la clasificación de ítems
# =============================================================================
matriz_similitud_interacciones = cosine_similarity(vector_por_usuario, matriz_items)
df_similitud_interacciones = pd.DataFrame(
    matriz_similitud_interacciones,
    index=vector_por_usuario.index,
    columns=matriz_items.index
)

COLOR_TITULO    = "\033[1;32m"  # Verde brillante
COLOR_ITEM      = "\033[1;34m"  # Azul brillante
COLOR_VAL       = "\033[1;33m"  # Amarillo brillante
COLOR_SIMILITUD = "\033[1;33m"  # Amarillo brillante para el score
COLOR_RESET     = "\033[0m"     # Reset

def star_rating(score, max_stars=5):
    """
    Convierte un score (0 a 1) en una cadena de estrellas.
    Se redondea el score a un máximo de 'max_stars' estrellas.
    """
    num_stars = int(round(score * max_stars))
    num_stars = min(num_stars, max_stars)
    return "★" * num_stars + "☆" * (max_stars - num_stars)

def normalize_series(s):
    """Normaliza una serie en el rango [0, 1]."""
    if s.max() == s.min():
        return s
    return (s - s.min()) / (s.max() - s.min())

def obtener_historial(usuarios_historico):
    """
    Retorna un diccionario con el historial de ítems visitados por cada usuario.
    Formato: {id_user: [lista de id_item]}.
    """
    return usuarios_historico.groupby('id_user')['id_item'].apply(list).to_dict()


def recomendar_con_enfoque_aditivo(user_id, alpha=0.33, beta=0.33, gamma=0.34, N=10):
    """
    Genera recomendaciones para un usuario combinando:
      - Similitud con sus preferencias.
      - Concordancia con su historial de interacciones.
      - Popularidad (visitas) del ítem.
    
    Además muestra:
      - Gustos o preferencias del usuario.
      - Ítems ya visitados (con valoración y visitas).
      - Recomendaciones con una explicación detallada.
    
    Retorna una lista de tuplas: (nombre_item, score_final)
    """
    # Extraer scores:
    score_pref = matriz_similitud_items_df.loc[user_id]
    score_hist = df_similitud_interacciones.loc[user_id]
    score_vis = items_names.set_index('id_item')['visitas'].apply(lambda x: np.log(1 + x))
    
    # Conservar solo los ítems presentes en las tres fuentes
    common_items = score_pref.index.intersection(score_hist.index).intersection(score_vis.index)
    score_pref = score_pref.loc[common_items]
    score_hist = score_hist.loc[common_items]
    score_vis = score_vis.loc[common_items]
    
    # Normalizar cada score
    score_pref_norm = normalize_series(score_pref)
    score_hist_norm = normalize_series(score_hist)
    score_vis_norm = normalize_series(score_vis)
    
    # Combinar scores de forma aditiva con pesos
    final_score = alpha * score_pref_norm + beta * score_hist_norm + gamma * score_vis_norm
    
    # Excluir ítems ya visitados
    historial = obtener_historial(usuarios_historico)
    items_visitados = historial.get(user_id, [])
    final_score = final_score.drop(labels=items_visitados, errors='ignore')
    
    # Seleccionar los top N ítems
    top_items = final_score.nlargest(N)
    
    # Mostrar información del usuario
    print(f"\n{COLOR_TITULO}Gustos o preferencias del usuario:{COLOR_RESET}")
    if user_id in matriz_filtrada.columns:
        gustos_usuario = matriz_filtrada[user_id]
        if 'padre' in gustos_usuario.index:
            gustos_usuario = gustos_usuario.drop('padre')
        gustos_usuario = gustos_usuario[gustos_usuario > 0]
        if not gustos_usuario.empty:
            gustos_usuario_df = pd.DataFrame(gustos_usuario).rename(columns={user_id: 'score'})
            gustos_usuario_df.index.name = 'id_preferencia'
            padres_indexed = padres.set_index('id')
            gustos_usuario_df = gustos_usuario_df.merge(padres_indexed, left_index=True, right_index=True, how='left')
            for idx, row in gustos_usuario_df.iterrows():
                print(f"{COLOR_ITEM}{row['name']}{COLOR_RESET} (id_preferencia: {idx}) - Score: {COLOR_VAL}{row['score']}{COLOR_RESET}")
    
    print(f"\n{COLOR_TITULO}Items visitados por el usuario:{COLOR_RESET}")
    for item in items_visitados:
        item_name = items_names.loc[items_names['id_item'] == item, 'name_item'].values[0]
        valoracion = usuarios_historico.loc[(usuarios_historico['id_user'] == user_id) & (usuarios_historico['id_item'] == item),
                                             'valoracion'].values[0]
        visitas = items_names.loc[items_names['id_item'] == item, 'visitas'].values[0]
        print(f"{COLOR_ITEM}{item_name}{COLOR_RESET} - {COLOR_VAL}Valoración: {valoracion}{COLOR_RESET} - {COLOR_VAL}Visitas: {visitas}{COLOR_RESET}")
    
    print(f"\n{COLOR_TITULO}Recomendaciones:{COLOR_RESET}")
    for item_id, score in top_items.items():
        item_name = items_names.loc[items_names['id_item'] == item_id, 'name_item'].values[0]
        visitas = items_names.loc[items_names['id_item'] == item_id, 'visitas'].values[0]
        sim_score = score_pref_norm.loc[item_id]
        hist_score = score_hist_norm.loc[item_id]
        vis_score = score_vis_norm.loc[item_id]
        
        sim_stars = star_rating(sim_score)
        hist_stars = star_rating(hist_score)
        vis_stars = star_rating(vis_score)
        
        print(f"{COLOR_TITULO}Recomendado:{COLOR_RESET} {COLOR_ITEM}{item_name}{COLOR_RESET} {COLOR_SIMILITUD}(Score final: {score:.4f}){COLOR_RESET} - {COLOR_VAL}Visitas: {visitas}{COLOR_RESET}")
        print(f"  -> Explicación:")
        print(f"     Similitud con tus preferencias: {sim_score:.4f} {sim_stars}")
        print(f"     Concordancia con tu historial: {hist_score:.4f} {hist_stars}")
        print(f"     Popularidad (visitas):         {vis_score:.4f} {vis_stars}")
    
    recomendaciones = [
        (items_names.loc[items_names['id_item'] == item_id, 'name_item'].values[0], score)
        for item_id, score in top_items.items()
    ]
    return recomendaciones

user_id = 124
recomendaciones = recomendar_con_enfoque_aditivo(user_id, alpha=1/3, beta=1/3, gamma=1/3, N=5)


Gustos o preferencias del usuario:
Prehistórico (id_preferencia: 16) - Score: 10.0
Árabe (id_preferencia: 17) - Score: 9.0
Jardines botánicos (id_preferencia: 29) - Score: 10.0
Teatros (id_preferencia: 38) - Score: 8.0
Eventos (id_preferencia: 46) - Score: 9.0
Tiendas tradicionales (id_preferencia: 48) - Score: 9.0
Exposiciones (id_preferencia: 50) - Score: 9.0
Nauticos (id_preferencia: 57) - Score: 5.0
Arte (id_preferencia: 62) - Score: 9.0
Catedrales (id_preferencia: 72) - Score: 9.0
Claustros (id_preferencia: 76) - Score: 10.0
Sinagogas (id_preferencia: 77) - Score: 8.0
Puertas (id_preferencia: 90) - Score: 9.0
Centro histórico (id_preferencia: 93) - Score: 9.0
Atarazanas (id_preferencia: 100) - Score: 10.0
Esculturas (id_preferencia: 105) - Score: 2.0
Centros de salud y spa (id_preferencia: 108) - Score: 4.0

Items visitados por el usuario:
Centro Julio Gonzalez-IVAM - Valoración: 6 - Visitas: 23
El Oceanográfico - Valoración: 7 - Visitas: 35
Museo Taurino - Valoración: 4 - Visita

$$
\mathrm{Score}_{\mathrm{final}}(i) \;=\; \alpha \,\cdot\, \mathrm{score}_{\mathrm{pref\_norm}}(i)
\;+\;\beta \,\cdot\, \mathrm{score}_{\mathrm{hist\_norm}}(i)
\;+\;\gamma \,\cdot\, \mathrm{score}_{\mathrm{vis\_norm}}(i)
$$

donde:

- $\mathrm{score}_{\mathrm{pref\_norm}}(i)$ es la similitud normalizada entre las preferencias del usuario y el ítem $i$.
- $\mathrm{score}_{\mathrm{hist\_norm}}(i)$ es la concordancia normalizada basada en el historial de interacciones del usuario para el ítem $i$.
- $\mathrm{score}_{\mathrm{vis\_norm}}(i)$ es la popularidad normalizada (calculada a partir de las visitas) del ítem $i$.
- $\alpha$, $\beta$ y $\gamma$ son los pesos asignados a cada componente, con la condición:

$$
\alpha \;+\; \beta \;+\; \gamma \;=\; 1.
$$
